# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

I audited the FlyRank research paper (`docs/flyrank-seo-research-march-2026.pdf`) first, and the audit gave me
three questions that generalize to my own Week-5 model. This notebook applies them to myself:

1. Where does the **label** come from — and is the validation design built to carry the claim?
2. Does the **split** respect the data's real structure (whole clients, not random pages)?
3. What would a **leak** actually do to my numbers if one slipped in?

Sections 1-3 are the audit; section 4 rewrites my own boldest sentence in safe language. The w04/w05 numbers I
am re-auditing are in `work/outputs/w04_baseline_metrics.json` and `work/outputs/w05_model_metrics.json`.

## 1. Two paper findings + my methodology questions

The paper's evidence standard is genuinely careful — "direct aggregate comparisons lead", ML pages are
"exploratory appendix material", weaker internal constructs get demoted (pp. 4-5). My questions are about the
parts it does *not* print, which is exactly what my own notebook should print.

**Finding A — "Growing content is longer, younger, and positioned slightly better."** *(Finding #1, CONFIRMED)*

- **As printed:** rising pages average 3.2K vs 2.3K words (37.6% longer) and 184 vs 230 days old, over large
  cohorts (74.2K rising vs 45.3K falling).
- **Where does the label come from?** `trend_direction` is a **same-window** 30d-vs-prev-30d impression change
  (p.5: Up > +10%, Down > -10%). The up/down label and the profiling features (words, age, position, health) live
  in the **same window** — no time gap, so this is a descriptive cross-section, not prediction.
- **Does the validation carry the claim?** For the descriptive profile, yes — it is honestly called an
  observational comparison (p.6) and the evidence standard demotes weaker constructs. What it cannot carry is the
  "What to Do With This" advice (expand thin pages, refresh quarterly): a same-window contrast cannot prove that
  refreshing *causes* retention.
- **My questions:** (1) Would the profile survive if "growing" meant *next*-period growth? (2) Counts are shown
  but no base rate — what share of the portfolio was up vs down? (3) Position is embedded in the health metric
  in the same panel, so part of the "position" difference is the metric colliding with itself.

**How I answered in my lane.** My label is `declined_next_30d = (apr_imp < 0.8 * imp_march)` — an April (future)
observation against March-only features. The window trap the paper's trend label sits on is structurally absent
from mine. Cell 1a below proves the windows are adjacent; 1b prints the base rates the paper omits; 1c shows the
split difference before we even train.

**Finding B — "Logistic regression: 71% holdout accuracy predicting growing vs declining pages."** *(ML appendix,
Growth & Classification)*

- **As printed:** 71% holdout accuracy (p.29); the same appendix reports Random Forest "feature importance for
  health score" with Average Position 43% and Impressions 32% (p.27).
- **Where does the label come from?** The same **same-window** `trend_direction` buckets again — label and
  features share one window.
- **Does the validation carry the claim?** Four things the appendix does not print: **(1) base rate** — accuracy
  is unreadable on an imbalanced label; a "predict the majority class" rule can score 60-71% with zero signal.
  **(2) grouping** — 57 brands in the portfolio; a plain random holdout puts the same brand's pages on both
  sides, letting the model memorize brand-level noise. **(3) shared construct** — the health-score importance is
  partly mechanical, because health = impressions (30) + position (30) + CTR (20) + scroll (20); the paper itself
  flags it ("partly constructed from... these inputs"). **(4) metric choice** — accuracy picks a threshold;
  a ranking task is better read with precision@K + ROC-AUC.
- **My questions, condensed:** what base rate, what kind of holdout, and which columns feed the label?

**How I answered in my lane.** P@K + ROC-AUC with base rates printed for train and held-out; whole clients held
out (30 train / 13 held-out, overlap 0); permutation importance measured on held-out pages; label never derives
from the features. The rest of this notebook re-runs my own w05 model under exactly those three questions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Same data, same features, same Random Forest (400 trees / `min_samples_leaf=15` / `sqrt` / seed 42). **Only the
split changes:**

| split | what it does | why it is on this list |
|---|---|---|
| **BEFORE — random 70/30, stratified** | test pages come from the same clients that trained the model (overlap 39/39 here) | the naive default, and the plausible reading of the paper's "holdout accuracy" |
| **AFTER — grouped by client** (30 train / 13 held-out) | a whole set of clients is never seen | simulates "score a NEW client's pages" — this is what w05 shipped |

w05 kept this same grouped split for the rule, LR and RF. Here I re-fit only the RF because the question is
"does the split change the answer?" — and I add an after-after GroupKFold so one lucky 30/13 draw cannot carry
the conclusion.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

The same hunt from Week 3, on the **final** feature set. Two leak families:

1. **Time window** — every feature must end on 2026-03-31. One tempting dimension is deliberately rejected:
   staleness from `dim_content.content_updated_date` is an *as-of-release* snapshot (many rows carry
   2026-07-01 dates), so it is NOT knowable at the decision moment — it would smuggle in the future.
2. **Label reach** — `declined_next_30d` and its source `apr_imp` must never ride along as features.

The cell asserts all of it, then proves the trap is real: adding April impressions to the matrix shows exactly the
score jump an accidental leak would have bought.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

Reading my w05 error-and-interpretation section back with the paper-audit glasses on, the bold sentences were in
the family of "the model *is* good and *leans* on momentum". Fine for a working notebook, too loose as a submitted
claim. Rewrites below follow one rule the paper taught me: **observed, measured, directional, decision-support** —
never causal, never global, and never a forecast.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed locally with the month cache)
- [x] No client names, URLs, or private queries anywhere — only hashes and aggregates
- [x] Base rates printed next to every P@K; grouped split proven via before/after overlap counts
- [x] Leakage hunt re-run on the final feature set, with a demonstrated leak trap
- [x] My claims re-written in safe language: observed, measured, directional, decision-support
- [x] Metrics receipt written to `work/outputs/w06_validation_metrics.json`
- [x] Committed to my repo under `work/notebooks/` — then submit repo URL on the card. Done.